In [1]:
# Cell 1: tools
# ------------------------------------------------------------
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, Markdown, clear_output

# ---------- basis states ----------
zero = np.array([1.0, 0.0], dtype=complex)
one  = np.array([0.0, 1.0], dtype=complex)

ket00 = np.kron(zero, zero)
ket01 = np.kron(zero, one)
ket10 = np.kron(one,  zero)
ket11 = np.kron(one,  one)

# ---------- helpers ----------
def normalize(v):
    nrm = np.linalg.norm(v)
    if nrm <= 1e-15:
        return np.zeros_like(v, dtype=complex)
    return np.asarray(v, dtype=complex) / nrm

def wrap_phase_deg(phi_deg):
    # keep phase in [-180, 180)
    return ((phi_deg + 180.0) % 360.0) - 180.0

def complex_str(z, digits=3):
    zr = float(np.real(z))
    zi = float(np.imag(z))
    if abs(zi) < 10**(-digits):
        return f"{zr:.{digits}f}"
    if abs(zr) < 10**(-digits):
        return f"{zi:.{digits}f}i"
    sign = "+" if zi >= 0 else "-"
    return f"{zr:.{digits}f} {sign} {abs(zi):.{digits}f}i"

def ket_to_string_2q(ket, digits=3):
    labels = ["|00⟩", "|01⟩", "|10⟩", "|11⟩"]
    pieces = []
    for amp, label in zip(np.asarray(ket).reshape(-1), labels):
        if abs(amp) > 1e-10:
            pieces.append(f"({complex_str(amp, digits)}) {label}")
    return " + ".join(pieces) if pieces else "0"

def fidelity_pure(ket_a, ket_b):
    ket_a = normalize(np.asarray(ket_a, dtype=complex).reshape(-1))
    ket_b = normalize(np.asarray(ket_b, dtype=complex).reshape(-1))
    if ket_a.size != 4 or ket_b.size != 4:
        raise ValueError("Expected 2-qubit state vectors of length 4.")
    if np.linalg.norm(ket_a) <= 1e-15 or np.linalg.norm(ket_b) <= 1e-15:
        return 0.0
    return float(abs(np.vdot(ket_a, ket_b)) ** 2)

# ---------- source states ----------
def source_state(alpha_deg=45.0, phase_deg=0.0):
    """
    |psi(alpha,phi)> = cos(alpha)|00> + exp(i phi) sin(alpha)|11>
    alpha in degrees
    phase in degrees
    """
    a = np.deg2rad(alpha_deg)
    ph = np.deg2rad(phase_deg)
    ket = np.cos(a) * ket00 + np.exp(1j * ph) * np.sin(a) * ket11
    return normalize(ket)

def noisy_source_state(
    alpha_deg=45.0,
    phase_deg=0.0,
    alpha_jitter_deg=0.0,
    phase_jitter_deg=0.0,
    source_amp_noise=0.0,
):
    """
    Pure-state effective source noise.
    Start from the restricted |00>/|11> source, then perturb amplitudes/phases
    and renormalize, staying inside the same sector.
    """
    a = np.deg2rad(alpha_deg)
    ph = np.deg2rad(phase_deg)

    c00 = np.cos(a)
    c11 = np.exp(1j * ph) * np.sin(a)

    # original teaching jitter retained
    if alpha_jitter_deg > 0.0:
        a = np.deg2rad(np.clip(alpha_deg + random.gauss(0.0, alpha_jitter_deg), 0.0, 90.0))
        c00 = np.cos(a)
        c11 = np.exp(1j * ph) * np.sin(a)

    if phase_jitter_deg > 0.0:
        ph = np.deg2rad(wrap_phase_deg(phase_deg + random.gauss(0.0, phase_jitter_deg)))
        # keep magnitudes from current a
        c00 = np.cos(a)
        c11 = np.exp(1j * ph) * np.sin(a)

    # new amplitude/state-quality perturbation
    if source_amp_noise > 0.0:
        re00 = random.gauss(0.0, source_amp_noise)
        im00 = random.gauss(0.0, source_amp_noise)
        re11 = random.gauss(0.0, source_amp_noise)
        im11 = random.gauss(0.0, source_amp_noise)

        c00 = c00 + (re00 + 1j * im00)
        c11 = c11 + (re11 + 1j * im11)

    ket = c00 * ket00 + c11 * ket11
    return normalize(ket)

# ---------- Bell states on the hub pair (qubits 2 and 3) ----------
BELL_KETS = {
    "Phi+": normalize(ket00 + ket11),
    "Phi-": normalize(ket00 - ket11),
    "Psi+": normalize(ket01 + ket10),
    "Psi-": normalize(ket01 - ket10),
}

# ---------- noisy Bell-state projector ----------
def noisy_bell_state(bell_branch="Phi+", bsm_noise=0.0):
    """
    Pure-state effective BSM noise:
    use the selected Bell state plus random admixture of the other Bell states,
    then renormalize.
    """
    bell = np.array(BELL_KETS[bell_branch], dtype=complex)

    if bsm_noise <= 0.0:
        return bell

    mix = np.zeros(4, dtype=complex)
    for label, ket in BELL_KETS.items():
        if label == bell_branch:
            continue
        coeff = random.gauss(0.0, bsm_noise) + 1j * random.gauss(0.0, bsm_noise)
        mix += coeff * ket

    noisy = bell + mix
    return normalize(noisy)

# ---------- pure-state concurrence ----------
def concurrence_pure(ket_2q):
    """
    For |psi> = a|00> + b|01> + c|10> + d|11>,
    concurrence = 2 |ad - bc|
    """
    ket_2q = normalize(np.asarray(ket_2q, dtype=complex).reshape(-1))
    if ket_2q.size != 4:
        raise ValueError("Expected a 2-qubit state vector of length 4.")
    a, b, c, d = ket_2q
    return float(2.0 * abs(a * d - b * c))

# ---------- direct state-vector projection ----------
def project_onto_branch(psi1234, bell_branch="Phi+", bsm_noise=0.0):
    """
    Project qubits 2 and 3 of a 4-qubit pure state onto a Bell-like state.
    Qubit order: (1,2,3,4)

    Returns:
        p_branch, psi14_conditional, bell_used
    """
    bell = noisy_bell_state(bell_branch=bell_branch, bsm_noise=bsm_noise).reshape(2, 2)
    psi_tensor = np.asarray(psi1234, dtype=complex).reshape(2, 2, 2, 2)

    # Contract Bell bra on hub qubits (2,3) => leaves remote qubits (1,4)
    remote = np.tensordot(np.conj(bell), psi_tensor, axes=([0, 1], [1, 2]))  # shape (2,2)
    remote = remote.reshape(4)

    p_branch = float(np.vdot(remote, remote).real)
    psi14 = normalize(remote) if p_branch > 1e-15 else np.zeros(4, dtype=complex)

    return p_branch, psi14, bell.reshape(4)

# ---------- one swap run ----------
def simulate_swap_state(
    alpha_A_deg=45.0,
    phase_A_deg=0.0,
    alpha_B_deg=45.0,
    phase_B_deg=0.0,
    bell_branch="Phi+",
    alpha_jitter_deg=0.0,
    phase_jitter_deg=0.0,
    source_amp_noise=0.0,
    bsm_noise=0.0,
    target_psi14=None,
):
    psi12 = noisy_source_state(
        alpha_deg=alpha_A_deg,
        phase_deg=phase_A_deg,
        alpha_jitter_deg=alpha_jitter_deg,
        phase_jitter_deg=phase_jitter_deg,
        source_amp_noise=source_amp_noise,
    )
    psi34 = noisy_source_state(
        alpha_deg=alpha_B_deg,
        phase_deg=phase_B_deg,
        alpha_jitter_deg=alpha_jitter_deg,
        phase_jitter_deg=phase_jitter_deg,
        source_amp_noise=source_amp_noise,
    )

    psi1234 = np.kron(psi12, psi34)

    p_branch, psi14, bell_used = project_onto_branch(
        psi1234,
        bell_branch=bell_branch,
        bsm_noise=bsm_noise,
    )
    C = concurrence_pure(psi14) if p_branch > 1e-15 else 0.0

    if target_psi14 is None:
        target_psi14 = psi14.copy()

    F = fidelity_pure(psi14, target_psi14) if p_branch > 1e-15 else 0.0

    return {
        "psi12": psi12,
        "psi34": psi34,
        "psi1234": psi1234,
        "branch": bell_branch,
        "bell_used": bell_used,
        "p_branch": p_branch,
        "psi14": psi14,
        "target_psi14": normalize(target_psi14),
        "concurrence": C,
        "fidelity": F,
    }

# ---------- simple teaching noise model ----------
def noisy_parameters(
    alpha_A_deg,
    phase_A_deg,
    alpha_B_deg,
    phase_B_deg,
    alpha_jitter_deg=0.0,
    phase_jitter_deg=0.0,
):
    aA = np.clip(alpha_A_deg + random.gauss(0.0, alpha_jitter_deg), 0.0, 90.0)
    pA = wrap_phase_deg(phase_A_deg + random.gauss(0.0, phase_jitter_deg))
    aB = np.clip(alpha_B_deg + random.gauss(0.0, alpha_jitter_deg), 0.0, 90.0)
    pB = wrap_phase_deg(phase_B_deg + random.gauss(0.0, phase_jitter_deg))
    return aA, pA, aB, pB

def simulate_swap_noisy(
    alpha_A_deg=45.0,
    phase_A_deg=0.0,
    alpha_B_deg=45.0,
    phase_B_deg=0.0,
    bell_branch="Phi+",
    alpha_jitter_deg=0.0,
    phase_jitter_deg=0.0,
    source_amp_noise=0.0,
    bsm_noise=0.0,
    nshots=200,
):
    if nshots <= 0:
        raise ValueError("nshots must be positive.")

    probs = []
    concs = []
    fids = []

    # local ideal target: same settings, but no noise
    target_sim = simulate_swap_state(
        alpha_A_deg=alpha_A_deg,
        phase_A_deg=phase_A_deg,
        alpha_B_deg=alpha_B_deg,
        phase_B_deg=phase_B_deg,
        bell_branch=bell_branch,
        alpha_jitter_deg=0.0,
        phase_jitter_deg=0.0,
        source_amp_noise=0.0,
        bsm_noise=0.0,
        target_psi14=None,
    )
    target_psi14 = target_sim["psi14"]

    last_clean = None
    for _ in range(nshots):
        sim = simulate_swap_state(
            alpha_A_deg=alpha_A_deg,
            phase_A_deg=phase_A_deg,
            alpha_B_deg=alpha_B_deg,
            phase_B_deg=phase_B_deg,
            bell_branch=bell_branch,
            alpha_jitter_deg=alpha_jitter_deg,
            phase_jitter_deg=phase_jitter_deg,
            source_amp_noise=source_amp_noise,
            bsm_noise=bsm_noise,
            target_psi14=target_psi14,
        )
        probs.append(sim["p_branch"])
        concs.append(sim["concurrence"])
        fids.append(sim["fidelity"])
        last_clean = sim

    return {
        "mean_p_branch": float(np.mean(probs)),
        "std_p_branch": float(np.std(probs)),
        "mean_concurrence": float(np.mean(concs)),
        "std_concurrence": float(np.std(concs)),
        "mean_fidelity": float(np.mean(fids)),
        "std_fidelity": float(np.std(fids)),
        "target_psi14": target_psi14,
        "nshots": int(nshots),
        "last_sim": last_clean,
        "all_p_branch": probs,
        "all_concurrence": concs,
        "all_fidelity": fids,
    }

# ---------- plot helpers ----------
def alpha_beta_grid(
    phase_A_deg=0.0,
    phase_B_deg=0.0,
    bell_branch="Phi+",
    metric="concurrence",
    alpha_jitter_deg=0.0,
    phase_jitter_deg=0.0,
    source_amp_noise=0.0,
    bsm_noise=0.0,
    nshots=100,
    n_alpha=61,
    n_beta=61,
):
    alphas = np.linspace(0.0, 90.0, n_alpha)
    betas  = np.linspace(0.0, 90.0, n_beta)
    Z = np.zeros((n_beta, n_alpha), dtype=float)

    for i, beta in enumerate(betas):
        for j, alpha in enumerate(alphas):
            if (
                alpha_jitter_deg > 0.0
                or phase_jitter_deg > 0.0
                or source_amp_noise > 0.0
                or bsm_noise > 0.0
            ):
                sim = simulate_swap_noisy(
                    alpha_A_deg=alpha,
                    phase_A_deg=phase_A_deg,
                    alpha_B_deg=beta,
                    phase_B_deg=phase_B_deg,
                    bell_branch=bell_branch,
                    alpha_jitter_deg=alpha_jitter_deg,
                    phase_jitter_deg=phase_jitter_deg,
                    source_amp_noise=source_amp_noise,
                    bsm_noise=bsm_noise,
                    nshots=nshots,
                )
                if metric == "concurrence":
                    Z[i, j] = sim["mean_concurrence"]
                elif metric == "probability":
                    Z[i, j] = sim["mean_p_branch"]
                elif metric == "fidelity":
                    Z[i, j] = sim["mean_fidelity"]
                else:
                    raise ValueError("metric must be 'concurrence', 'probability', or 'fidelity'")
            else:
                sim = simulate_swap_state(
                    alpha_A_deg=alpha,
                    phase_A_deg=phase_A_deg,
                    alpha_B_deg=beta,
                    phase_B_deg=phase_B_deg,
                    bell_branch=bell_branch,
                )
                if metric == "concurrence":
                    Z[i, j] = sim["concurrence"]
                elif metric == "probability":
                    Z[i, j] = sim["p_branch"]
                elif metric == "fidelity":
                    Z[i, j] = sim["fidelity"]
                else:
                    raise ValueError("metric must be 'concurrence', 'probability', or 'fidelity'")

    return alphas, betas, Z

def branch_bar_data(
    alpha_A_deg=45.0,
    phase_A_deg=0.0,
    alpha_B_deg=45.0,
    phase_B_deg=0.0,
    alpha_jitter_deg=0.0,
    phase_jitter_deg=0.0,
    source_amp_noise=0.0,
    bsm_noise=0.0,
    nshots=200,
    metric="concurrence",
):
    labels = ["Phi+", "Phi-", "Psi+", "Psi-"]
    values = []

    for branch in labels:
        if (
            alpha_jitter_deg > 0.0
            or phase_jitter_deg > 0.0
            or source_amp_noise > 0.0
            or bsm_noise > 0.0
        ):
            sim = simulate_swap_noisy(
                alpha_A_deg=alpha_A_deg,
                phase_A_deg=phase_A_deg,
                alpha_B_deg=alpha_B_deg,
                phase_B_deg=phase_B_deg,
                bell_branch=branch,
                alpha_jitter_deg=alpha_jitter_deg,
                phase_jitter_deg=phase_jitter_deg,
                source_amp_noise=source_amp_noise,
                bsm_noise=bsm_noise,
                nshots=nshots,
            )
            if metric == "concurrence":
                values.append(sim["mean_concurrence"])
            elif metric == "probability":
                values.append(sim["mean_p_branch"])
            elif metric == "fidelity":
                values.append(sim["mean_fidelity"])
            else:
                raise ValueError("metric must be 'concurrence', 'probability', or 'fidelity'")
        else:
            sim = simulate_swap_state(
                alpha_A_deg=alpha_A_deg,
                phase_A_deg=phase_A_deg,
                alpha_B_deg=alpha_B_deg,
                phase_B_deg=phase_B_deg,
                bell_branch=branch,
            )
            if metric == "concurrence":
                values.append(sim["concurrence"])
            elif metric == "probability":
                values.append(sim["p_branch"])
            elif metric == "fidelity":
                values.append(sim["fidelity"])
            else:
                raise ValueError("metric must be 'concurrence', 'probability', or 'fidelity'")

    return labels, values

# ---------- shared notebook state ----------
SWAP_DEMO = {
    "params": {
        "alpha_A_deg": 45.0,
        "phase_A_deg": 0.0,
        "alpha_B_deg": 45.0,
        "phase_B_deg": 0.0,
        "bell_branch": "Phi+",
        "alpha_jitter_deg": 0.0,
        "phase_jitter_deg": 0.0,
        "source_amp_noise": 0.0,
        "bsm_noise": 0.0,
        "nshots": 200,
    },
    "single_sim": None,
    "noisy_sim": None,
}

In [2]:
# Cell 2: ui
# ------------------------------------------------------------
alpha_A = widgets.FloatSlider(
    value=45.0, min=0.0, max=90.0, step=1.0,
    description="α_A", continuous_update=False
)
phase_A = widgets.FloatSlider(
    value=0.0, min=-180.0, max=180.0, step=5.0,
    description="φ_A", continuous_update=False
)
alpha_B = widgets.FloatSlider(
    value=45.0, min=0.0, max=90.0, step=1.0,
    description="α_B", continuous_update=False
)
phase_B = widgets.FloatSlider(
    value=0.0, min=-180.0, max=180.0, step=5.0,
    description="φ_B", continuous_update=False
)

bell_branch = widgets.Dropdown(
    options=["Phi+", "Phi-", "Psi+", "Psi-"],
    value="Phi+",
    description="Branch"
)

alpha_jitter = widgets.FloatSlider(
    value=0.0, min=0.0, max=8.0, step=0.2,
    description="Δα noise", continuous_update=False
)
phase_jitter = widgets.FloatSlider(
    value=0.0, min=0.0, max=40.0, step=1.0,
    description="Δφ noise", continuous_update=False
)

source_amp_noise = widgets.FloatSlider(
    value=0.0, min=0.0, max=0.50, step=0.01,
    description="Source noise", continuous_update=False
)
bsm_noise = widgets.FloatSlider(
    value=0.0, min=0.0, max=0.50, step=0.01,
    description="BSM noise", continuous_update=False
)

nshots = widgets.IntSlider(
    value=200, min=20, max=1000, step=20,
    description="Shots", continuous_update=False
)

run_button = widgets.Button(description="Run simulator", button_style="primary")
reset_button = widgets.Button(description="Reset", button_style="")

out_header = widgets.Output()
out_state = widgets.Output()
out_metrics = widgets.Output()
out_note = widgets.Output()

def store_params():
    SWAP_DEMO["params"] = {
        "alpha_A_deg": alpha_A.value,
        "phase_A_deg": phase_A.value,
        "alpha_B_deg": alpha_B.value,
        "phase_B_deg": phase_B.value,
        "bell_branch": bell_branch.value,
        "alpha_jitter_deg": alpha_jitter.value,
        "phase_jitter_deg": phase_jitter.value,
        "source_amp_noise": source_amp_noise.value,
        "bsm_noise": bsm_noise.value,
        "nshots": nshots.value,
    }

def render_ui():
    store_params()
    p = SWAP_DEMO["params"]

    clean = simulate_swap_state(
        alpha_A_deg=p["alpha_A_deg"],
        phase_A_deg=p["phase_A_deg"],
        alpha_B_deg=p["alpha_B_deg"],
        phase_B_deg=p["phase_B_deg"],
        bell_branch=p["bell_branch"],
        alpha_jitter_deg=0.0,
        phase_jitter_deg=0.0,
        source_amp_noise=0.0,
        bsm_noise=0.0,
    )
    noisy = simulate_swap_noisy(
        alpha_A_deg=p["alpha_A_deg"],
        phase_A_deg=p["phase_A_deg"],
        alpha_B_deg=p["alpha_B_deg"],
        phase_B_deg=p["phase_B_deg"],
        bell_branch=p["bell_branch"],
        alpha_jitter_deg=p["alpha_jitter_deg"],
        phase_jitter_deg=p["phase_jitter_deg"],
        source_amp_noise=p["source_amp_noise"],
        bsm_noise=p["bsm_noise"],
        nshots=p["nshots"],
    )

    SWAP_DEMO["single_sim"] = clean
    SWAP_DEMO["noisy_sim"] = noisy

    with out_header:
        clear_output()
        display(Markdown(
            "### Entanglement Swapper Demo\n"
            "Two source pairs feed a hub Bell-state measurement. "
            "A selected Bell branch heralds a remote state on qubits 1 and 4."
        ))

    with out_state:
        clear_output()
        psi12_str = ket_to_string_2q(clean["psi12"])
        psi34_str = ket_to_string_2q(clean["psi34"])
        psi14_str = ket_to_string_2q(clean["psi14"])
        target14_str = ket_to_string_2q(clean["target_psi14"])

        display(Markdown(
            f"**Ideal Source A:**  \n{psi12_str}\n\n"
            f"**Ideal Source B:**  \n{psi34_str}\n\n"
            f"**Selected Bell branch:** `{p['bell_branch']}`\n\n"
            f"**Ideal swapped remote state:**  \n{psi14_str}\n\n"
            f"**Target remote state used for fidelity:**  \n{target14_str}"
        ))

    with out_metrics:
        clear_output()
        html = f"""
        <div style="padding:10px; border:1px solid #ddd; border-radius:8px;">
          <b>Ideal single-run metrics</b><br>
          Branch probability = {clean['p_branch']:.4f}<br>
          Concurrence = {clean['concurrence']:.4f}<br>
          Fidelity to target = {clean['fidelity']:.4f}<br><br>

          <b>Noisy average metrics</b><br>
          Mean branch probability = {noisy['mean_p_branch']:.4f} ± {noisy['std_p_branch']:.4f}<br>
          Mean concurrence = {noisy['mean_concurrence']:.4f} ± {noisy['std_concurrence']:.4f}<br>
          Mean fidelity = {noisy['mean_fidelity']:.4f} ± {noisy['std_fidelity']:.4f}<br>
          Shots = {noisy['nshots']}
        </div>
        """
        display(HTML(html))

    with out_note:
        clear_output()
        if p["bell_branch"] in ("Psi+", "Psi-") and p["source_amp_noise"] <= 1e-12 and p["bsm_noise"] <= 1e-12:
            display(Markdown(
                "> **Teaching note:** with ideal source states restricted to the "
                "`|00⟩ / |11⟩` sector, the odd Bell branches `Psi+` and `Psi-` "
                "ideally have zero probability."
            ))
        else:
            display(Markdown(
                "> **Teaching note:** `Δα` and `Δφ` represent parameter jitter, while "
                "`Source noise` and `BSM noise` are stronger effective dials for imperfect "
                "state preparation and imperfect Bell-state measurement. All runs remain "
                "pure-state realizations; the notebook averages the resulting metrics over many shots."
            ))

def on_run_clicked(_):
    render_ui()

def on_reset_clicked(_):
    alpha_A.value = 45.0
    phase_A.value = 0.0
    alpha_B.value = 45.0
    phase_B.value = 0.0
    bell_branch.value = "Phi+"
    alpha_jitter.value = 0.0
    phase_jitter.value = 0.0
    source_amp_noise.value = 0.0
    bsm_noise.value = 0.0
    nshots.value = 200
    render_ui()

run_button.on_click(on_run_clicked)
reset_button.on_click(on_reset_clicked)

controls_left = widgets.VBox([alpha_A, phase_A, alpha_B, phase_B])
controls_mid = widgets.VBox([bell_branch, alpha_jitter, phase_jitter, nshots])
controls_right = widgets.VBox([source_amp_noise, bsm_noise])
buttons = widgets.HBox([run_button, reset_button])

display(
    widgets.VBox([
        widgets.HBox([controls_left, controls_mid, controls_right]),
        buttons,
        out_header,
        out_state,
        out_metrics,
        out_note,
    ])
)

render_ui()

In [3]:
# Cell 3: plots
# ------------------------------------------------------------
plot_type = widgets.Dropdown(
    options=[
        "Concurrence heatmap",
        "Branch probability heatmap",
        "Fidelity heatmap",
        "Line plot vs α_A",
        "Bar chart by Bell branch",
    ],
    value="Concurrence heatmap",
    description="Plot"
)

metric_line = widgets.Dropdown(
    options=["concurrence", "probability", "fidelity"],
    value="concurrence",
    description="Line metric"
)

metric_bar = widgets.Dropdown(
    options=["concurrence", "probability", "fidelity"],
    value="concurrence",
    description="Bar metric"
)

line_alpha_B = widgets.FloatSlider(
    value=45.0, min=0.0, max=90.0, step=1.0,
    description="Fixed α_B", continuous_update=False
)

grid_points = widgets.IntSlider(
    value=41, min=21, max=81, step=10,
    description="Grid", continuous_update=False
)

plot_button = widgets.Button(description="Make plot", button_style="primary")
plot_out = widgets.Output()

def current_params():
    return SWAP_DEMO["params"].copy()

def make_plot(_=None):
    with plot_out:
        clear_output()
        p = current_params()

        if plot_type.value == "Concurrence heatmap":
            alphas, betas, Z = alpha_beta_grid(
                phase_A_deg=p["phase_A_deg"],
                phase_B_deg=p["phase_B_deg"],
                bell_branch=p["bell_branch"],
                metric="concurrence",
                alpha_jitter_deg=p["alpha_jitter_deg"],
                phase_jitter_deg=p["phase_jitter_deg"],
                source_amp_noise=p["source_amp_noise"],
                bsm_noise=p["bsm_noise"],
                nshots=p["nshots"],
                n_alpha=grid_points.value,
                n_beta=grid_points.value,
            )
            plt.figure(figsize=(6.4, 5.2))
            plt.imshow(
                Z,
                origin="lower",
                extent=[alphas[0], alphas[-1], betas[0], betas[-1]],
                aspect="auto",
                vmin=0.0,
                vmax=1.0,
            )
            plt.colorbar(label="Concurrence")
            plt.xlabel("α_A (deg)")
            plt.ylabel("α_B (deg)")
            plt.title(f"Swapped-state concurrence | branch = {p['bell_branch']}")
            plt.show()

        elif plot_type.value == "Branch probability heatmap":
            alphas, betas, Z = alpha_beta_grid(
                phase_A_deg=p["phase_A_deg"],
                phase_B_deg=p["phase_B_deg"],
                bell_branch=p["bell_branch"],
                metric="probability",
                alpha_jitter_deg=p["alpha_jitter_deg"],
                phase_jitter_deg=p["phase_jitter_deg"],
                source_amp_noise=p["source_amp_noise"],
                bsm_noise=p["bsm_noise"],
                nshots=p["nshots"],
                n_alpha=grid_points.value,
                n_beta=grid_points.value,
            )
            plt.figure(figsize=(6.4, 5.2))
            plt.imshow(
                Z,
                origin="lower",
                extent=[alphas[0], alphas[-1], betas[0], betas[-1]],
                aspect="auto",
                vmin=0.0,
                vmax=1.0,
            )
            plt.colorbar(label="Branch probability")
            plt.xlabel("α_A (deg)")
            plt.ylabel("α_B (deg)")
            plt.title(f"Heralding probability | branch = {p['bell_branch']}")
            plt.show()

        elif plot_type.value == "Fidelity heatmap":
            alphas, betas, Z = alpha_beta_grid(
                phase_A_deg=p["phase_A_deg"],
                phase_B_deg=p["phase_B_deg"],
                bell_branch=p["bell_branch"],
                metric="fidelity",
                alpha_jitter_deg=p["alpha_jitter_deg"],
                phase_jitter_deg=p["phase_jitter_deg"],
                source_amp_noise=p["source_amp_noise"],
                bsm_noise=p["bsm_noise"],
                nshots=p["nshots"],
                n_alpha=grid_points.value,
                n_beta=grid_points.value,
            )
            plt.figure(figsize=(6.4, 5.2))
            plt.imshow(
                Z,
                origin="lower",
                extent=[alphas[0], alphas[-1], betas[0], betas[-1]],
                aspect="auto",
                vmin=0.0,
                vmax=1.0,
            )
            plt.colorbar(label="Fidelity")
            plt.xlabel("α_A (deg)")
            plt.ylabel("α_B (deg)")
            plt.title(f"Local target fidelity | branch = {p['bell_branch']}")
            plt.show()

        elif plot_type.value == "Line plot vs α_A":
            xs = np.linspace(0.0, 90.0, 121)
            ys = []
            for aA in xs:
                if (
                    p["alpha_jitter_deg"] > 0.0
                    or p["phase_jitter_deg"] > 0.0
                    or p["source_amp_noise"] > 0.0
                    or p["bsm_noise"] > 0.0
                ):
                    sim = simulate_swap_noisy(
                        alpha_A_deg=aA,
                        phase_A_deg=p["phase_A_deg"],
                        alpha_B_deg=line_alpha_B.value,
                        phase_B_deg=p["phase_B_deg"],
                        bell_branch=p["bell_branch"],
                        alpha_jitter_deg=p["alpha_jitter_deg"],
                        phase_jitter_deg=p["phase_jitter_deg"],
                        source_amp_noise=p["source_amp_noise"],
                        bsm_noise=p["bsm_noise"],
                        nshots=p["nshots"],
                    )
                    if metric_line.value == "concurrence":
                        ys.append(sim["mean_concurrence"])
                    elif metric_line.value == "probability":
                        ys.append(sim["mean_p_branch"])
                    elif metric_line.value == "fidelity":
                        ys.append(sim["mean_fidelity"])
                    else:
                        raise ValueError("metric must be 'concurrence', 'probability', or 'fidelity'")
                else:
                    sim = simulate_swap_state(
                        alpha_A_deg=aA,
                        phase_A_deg=p["phase_A_deg"],
                        alpha_B_deg=line_alpha_B.value,
                        phase_B_deg=p["phase_B_deg"],
                        bell_branch=p["bell_branch"],
                    )
                    if metric_line.value == "concurrence":
                        ys.append(sim["concurrence"])
                    elif metric_line.value == "probability":
                        ys.append(sim["p_branch"])
                    elif metric_line.value == "fidelity":
                        ys.append(sim["fidelity"])
                    else:
                        raise ValueError("metric must be 'concurrence', 'probability', or 'fidelity'")

            plt.figure(figsize=(6.4, 4.4))
            plt.plot(xs, ys, linewidth=2)
            plt.xlabel("α_A (deg)")
            if metric_line.value == "concurrence":
                ylabel = "Concurrence"
            elif metric_line.value == "probability":
                ylabel = "Branch probability"
            else:
                ylabel = "Fidelity"
            plt.ylabel(ylabel)
            plt.ylim(0.0, 1.0)
            plt.title(
                f"{ylabel} vs α_A | fixed α_B = {line_alpha_B.value:.1f}°, branch = {p['bell_branch']}"
            )
            plt.grid(True, alpha=0.3)
            plt.show()

        elif plot_type.value == "Bar chart by Bell branch":
            labels, values = branch_bar_data(
                alpha_A_deg=p["alpha_A_deg"],
                phase_A_deg=p["phase_A_deg"],
                alpha_B_deg=p["alpha_B_deg"],
                phase_B_deg=p["phase_B_deg"],
                alpha_jitter_deg=p["alpha_jitter_deg"],
                phase_jitter_deg=p["phase_jitter_deg"],
                source_amp_noise=p["source_amp_noise"],
                bsm_noise=p["bsm_noise"],
                nshots=p["nshots"],
                metric=metric_bar.value,
            )
            plt.figure(figsize=(6.0, 4.2))
            plt.bar(labels, values)
            plt.ylim(0.0, 1.05)
            if metric_bar.value == "concurrence":
                ylabel = "Concurrence"
            elif metric_bar.value == "probability":
                ylabel = "Branch probability"
            else:
                ylabel = "Fidelity"
            plt.ylabel(ylabel)
            plt.title(f"{ylabel} by Bell branch")
            plt.show()

plot_button.on_click(make_plot)

display(
    widgets.VBox([
        widgets.HBox([plot_type, metric_line, metric_bar]),
        widgets.HBox([line_alpha_B, grid_points]),
        plot_button,
        plot_out,
    ])
)

make_plot()